Building a Chatbot with LangChain — Google Colab
This notebook follows the original lesson: chat model calls, message history, prompt templates, language-based prompts, and conversation-history trimming.
Run instructions
1. Open this notebook in Google Colab.
2. In the left sidebar, open Secrets (key icon).
3. When prompted by the setup cell, paste your newly generated Groq API key. Input is hidden.
4. Run the notebook from top to bottom.
The notebook securely prompts for your Groq API key (input is hidden) and uses Groq's openai/gpt-oss-20b chat model. If you prefer another model, change GROQ_MODEL in the model setup cell to a model ID available in your Groq account.

In [1]:
import os

groq_api_key = "*******"

os.environ["GROQ_API_KEY"] = groq_api_key

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [2]:
# Install the libraries used in this notebook
%pip -q install -U langchain langchain-core langchain-community langchain-groq tiktoken

print("Required libraries installed.")

Required libraries installed.


In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=groq_api_key,
    temperature=0
)

In [4]:
from langchain_groq import ChatGroq

# Current Groq model ID. Change this only if you select another
# active chat model from https://console.groq.com/docs/models
GROQ_MODEL = "openai/gpt-oss-20b"

model = ChatGroq(
    model=GROQ_MODEL,
    api_key=groq_api_key,
    temperature=0
)

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.5', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7eb9b197e990>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7eb9b197f390>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, trim_messages
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# In-memory store: history lasts only for the current Colab runtime.
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

/tmp/ipykernel_5981/3299612781.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [46]:
# HumanMessage is imported in the setup cell
model.invoke([HumanMessage(content="Hi , My name is Nikitaa and I am a Btech student at MSIT")])

AIMessage(content='Hello Nikitaa! 👋  \nIt’s great to meet a BTech student from MSIT. How can I help you today? Whether you’re looking for study tips, project ideas, career advice, or just want to chat about tech, I’m here for you.', additional_kwargs={'reasoning_content': 'User says "Hi , My name is Nikitaa and I am a Btech student at MSIT". They are greeting. We should respond politely, maybe ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 88, 'total_tokens': 189, 'completion_time': 0.104188061, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.00488494, 'prompt_tokens_details': None, 'queue_time': 0.07466442, 'total_time': 0.109073001}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e11c8bbf69', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a221f0-957c-7782-ad78-04ff3c3476e1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata=

In [47]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Nikitaa and I am a Btech student at MSIT"),
        AIMessage(content="Hello Nikitaa! It's nice to meet you. \n\nAs a btech student, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You’re Nikitaa, a BTech student at MSIT.  \nIn short, you’re studying engineering, learning new tech, and likely working on projects or coursework related to your major. If you’d like to share more about your specific interests or projects, I’d love to hear about them!', additional_kwargs={'reasoning_content': 'User says: "Hey What\'s my name and what do I do?" They want to know their name and what they do. We know from earlier: "Hi , My name is Nikitaa and I am a Btech student at MSIT". So answer: Nikitaa, Btech student at MSIT. Also maybe mention what they do: studying, maybe working on projects. We should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 155, 'total_tokens': 306, 'completion_time': 0.176044478, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.010647153, 'prompt_tokens_details': None, 'queue_time': 0.192677354, 'total_time': 0.186691631}, 'model_name': 'openai/gpt-oss-20b', 'sy

Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [51]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [53]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(
                content="Hi, my name is Nikitaa and I am a BTech student at MSIT"
            )
        ],
        "language": "English"
    },
    config=config
)

print(response.content)

Hello Nikitaa! It’s great to meet you. How can I help you today?


In [54]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ],
        "language": "English"
    },
    config=config
)

print(response.content)

I’m sorry, but I don’t have that information. Could you let me know your name?


In [55]:
response.content

'I’m sorry, but I don’t have that information. Could you let me know your name?'

In [65]:
response=with_message_history.invoke(
    {
        "messages": [
        HumanMessage(content="What's my name?")
        ],
     "language": "English"
     },
    config=config,
)

In [66]:
response.content

'I don’t have that information. Could you tell me your name?'

In [67]:
# Change the config: use a different session ID
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    {
        "messages": [

    HumanMessage(content="Whats my name")],
        "language": "English"
    },
    config=config1
)
response.content

'Your name is John.'

Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [68]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all questions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [73]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Nikitaa")]})

AIMessage(content='Hello Nikitaa! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says "Hi My name is Nikitaa". We should respond politely, maybe ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 95, 'total_tokens': 139, 'completion_time': 0.048683425, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.005450438, 'prompt_tokens_details': None, 'queue_time': 0.011197416, 'total_time': 0.054133863}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_92d51d08e5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a22208-7344-712e-89b8-df175f52ad19-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 44, 'total_tokens': 139, 'output_token_details': {'reasoning': 22}})

In [74]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [75]:
config = {"configurable": {"session_id": "chat3"}}
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi My name is Nikitaa")]},
    config=config
)
response.content

'Got it—your name is Nikitaa. How can I help you today?'

In [76]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config=config,
)
response.content

'Your name is Nikitaa.'

In [77]:
# Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [78]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Nikitaa")],"language":"Hindi"})
response.content

'नमस्ते निकीता! आपका स्वागत है। मैं आपकी किस प्रकार सहायता कर सकता/सकती हूँ?'

In [79]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [80]:
config = {"configurable": {"session_id": "chat4"}}
response=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Nikitaa")],"language":"Hindi"},
    config=config
)
response.content

'नमस्ते निकीता जी! आपका स्वागत है। आपसे मिलकर खुशी हुई। आप किस विषय में सहायता चाहते हैं?'

In [81]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [82]:
response.content

'आपका नाम निकीता है।'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [83]:
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [84]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model

)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I’m not sure—could you tell me a bit about your favorite flavors or ingredients? For example, do you prefer classic vanilla or something more adventurous like salted caramel or pistachio? Knowing a few of your likes (or dislikes) will help me suggest an ice‑cream that’s just right for you!'

In [85]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked for the sum of 2\u202f+\u202f2.'

In [86]:
# Let's wrap this in the message history
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}


In [87]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'I don’t actually know your name—could you let me know?'

In [88]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You haven’t asked me a math problem yet—so there isn’t one to recall. If you have a question in mind, feel free to share it!'

Vector Stores, Retrievers, and RAG — Google Colab
This notebook demonstrates LangChain Documents, Hugging Face embeddings, Chroma vector stores, similarity search, retrievers, and a basic RAG chain.
Run instructions
1. Open this notebook in Google Colab.
2. Run the library installation cell first.
3. Run the API setup cell and paste your new Groq API key when prompted. The input is hidden.
4. Run the remaining cells from top to bottom.
The embedding model all-MiniLM-L6-v2 is public and does not require a Hugging Face token for this example. The Groq API key is needed only for the chat model/RAG section.

Security: Do not share this notebook with an API key embedded in it or commit secrets to GitHub.

In [89]:
%pip -q install -U langchain langchain-core langchain-chroma langchain-groq langchain-huggingface sentence-transformers

print("All required libraries installed.")

All required libraries installed.


In [90]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:
- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
  The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.
Let's generate some sample documents:

In [91]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [92]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

# Enter your NEW Groq API key. Input is hidden.
groq_api_key = getpass("Enter your Groq API key: ").strip()

if not groq_api_key:
    raise ValueError("Groq API key cannot be empty.")

os.environ["GROQ_API_KEY"] = groq_api_key

llm = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-20b",
    temperature=0
)

llm

Enter your Groq API key: ··········


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.5', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7eb905d0f100>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7eb8fd2b82b0>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [93]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [94]:
%pip install -q -U --no-cache-dir \
    langchain langchain-core langchain-groq \
    langchain-huggingface langchain-chroma \
    chromadb sentence-transformers \
    opentelemetry-api opentelemetry-sdk

In [95]:
## VectorStores
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(documents,embedding=embeddings)
vectorstore


In [96]:
vectorstore.similarity_search("cat")  #sentence which has cat in it

[Document(id='c93c25e8-6a30-4994-95a6-dcf8af102499', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='19f82c98-1f4b-4400-81bc-3f363a3a7ae1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='9a387430-89ee-4298-be25-044355ddcc39', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='7b20b296-c109-413e-8828-3b587b27a8fe', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [97]:
## Async query
await vectorstore.asimilarity_search("cat")

[Document(id='c93c25e8-6a30-4994-95a6-dcf8af102499', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='19f82c98-1f4b-4400-81bc-3f363a3a7ae1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='9a387430-89ee-4298-be25-044355ddcc39', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='7b20b296-c109-413e-8828-3b587b27a8fe', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [98]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='c93c25e8-6a30-4994-95a6-dcf8af102499', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058602333069),
 (Document(id='19f82c98-1f4b-4400-81bc-3f363a3a7ae1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058602333069),
 (Document(id='9a387430-89ee-4298-be25-044355ddcc39', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='7b20b296-c109-413e-8828-3b587b27a8fe', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706)]

Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [99]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='19f82c98-1f4b-4400-81bc-3f363a3a7ae1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='9a387430-89ee-4298-be25-044355ddcc39', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [100]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])


[[Document(id='19f82c98-1f4b-4400-81bc-3f363a3a7ae1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='9a387430-89ee-4298-be25-044355ddcc39', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [101]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("tell me about dogs")
print(response.content)


Dogs are great companions, known for their loyalty and friendliness.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')